In [46]:
import pandas as pd
import numpy as np

In [47]:
df = pd.read_csv("spotify_tracks.csv")

In [48]:
df.head()

,track_id,track_name,artist_name,year,popularity,artwork_url,album_name,acousticness,danceability,duration_ms,...,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,track_url,language
0,2r0ROhr7pRN4MXDMT1fEmd,"Leo Das Entry (From ""Leo"")",Anirudh Ravichander,2024,59,https://i.scdn.co/image/ab67616d0000b273ce9c65...,"Leo Das Entry (From ""Leo"")",0.0241,0.753,97297.0,...,8.0,0.1000,-5.994,0.0,0.1030,110.997,4.0,0.459,https://open.spotify.com/track/2r0ROhr7pRN4MXD...,Tamil
1,4I38e6Dg52a2o2a8i5Q5PW,AAO KILLELLE,"Anirudh Ravichander, Pravin Mani, Vaishali Sri...",2024,47,https://i.scdn.co/image/ab67616d0000b273be1b03...,AAO KILLELLE,0.0851,0.780,207369.0,...,10.0,0.0951,-5.674,0.0,0.0952,164.995,3.0,0.821,https://open.spotify.com/track/4I38e6Dg52a2o2a...,Tamil
2,59NoiRhnom3lTeRFaBzOev,Mayakiriye Sirikiriye - Orchestral EDM,"Anirudh Ravichander, Anivee, Alvin Bruno",2024,35,https://i.scdn.co/image/ab67616d0000b27334a1dd...,Mayakiriye Sirikiriye (Orchestral EDM),0.0311,0.457,82551.0,...,2.0,0.0831,-8.937,0.0,0.1530,169.996,4.0,0.598,https://open.spotify.com/track/59NoiRhnom3lTeR...,Tamil
3,5uUqRQd385pvLxC8JX3tXn,Scene Ah Scene Ah - Experimental EDM Mix,"Anirudh Ravichander, Bharath Sankar, Kabilan, ...",2024,24,https://i.scdn.co/image/ab67616d0000b27332e623...,Scene Ah Scene Ah (Experimental EDM Mix),0.2270,0.718,115831.0,...,7.0,0.1240,-11.104,1.0,0.4450,169.996,4.0,0.362,https://open.spotify.com/track/5uUqRQd385pvLxC...,Tamil
4,1KaBRg2xgNeCljmyxBH1mo,Gundellonaa X I Am A Disco Dancer - Mashup,"Anirudh Ravichander, Benny Dayal, Leon James, ...",2024,22,https://i.scdn.co/image/ab67616d0000b2735a59b6...,Gundellonaa X I Am a Disco Dancer (Mashup),0.0153,0.689,129621.0,...,7.0,0.3450,-9.637,1.0,0.1580,128.961,4.0,0.593,https://open.spotify.com/track/1KaBRg2xgNeCljm...,Tamil


In [49]:
df[df["language"] == 'English'].shape

(23392, 22)

In [50]:
df[df["language"] == 'Hindi'].shape

(5740, 22)

In [51]:
df.columns

Index(['track_id', 'track_name', 'artist_name', 'year', 'popularity',
       'artwork_url', 'album_name', 'acousticness', 'danceability',
       'duration_ms', 'energy', 'instrumentalness', 'key', 'liveness',
       'loudness', 'mode', 'speechiness', 'tempo', 'time_signature', 'valence',
       'track_url', 'language'],
      dtype='str')

In [52]:
target_values = ["English", "Hindi"]
df_filtered = df[df["language"].isin(target_values)]

In [53]:
df_filtered.shape

(29132, 22)

In [54]:
feature_columns = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence"
]

print("Features used:")
for feature in feature_columns:
    print("-", feature)

Features used:
- acousticness
- danceability
- energy
- instrumentalness
- liveness
- loudness
- speechiness
- tempo
- valence


In [55]:
df_filtered.shape

(29132, 22)

In [56]:
df_filtered = df_filtered.drop_duplicates(
    subset="track_id"
).copy()

print("After removing duplicate track IDs:", df_filtered.shape)

After removing duplicate track IDs: (29129, 22)


In [58]:
df_filtered = df_filtered.drop_duplicates(
    subset=["track_name", "artist_name"],
    keep="first"
).copy()

print("Dataset shape after removing track + artist duplicates:", df_filtered.shape)

Dataset shape after removing track + artist duplicates: (18542, 22)


In [59]:
df_filtered = df_filtered.dropna(
    subset=feature_columns
).copy()

print("After removing missing feature values:", df_filtered.shape)

After removing missing feature values: (18542, 22)


In [60]:
df_filtered = df_filtered.reset_index(drop=True)

In [61]:
X = df_filtered[feature_columns].copy()

print("Feature matrix shape:", X.shape)
display(X.head())

Feature matrix shape: (18542, 9)


,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence
0,0.11900,0.801,0.504,0.0000,0.1320,-5.771,0.2590,119.971,0.821
1,0.28000,0.684,0.772,0.0052,0.0818,-8.282,0.0419,106.031,0.451
2,0.45100,0.609,0.491,0.0000,0.1110,-9.933,0.0945,99.186,0.718
3,0.02660,0.669,0.863,0.0000,0.3300,-3.364,0.1190,93.016,0.486
4,0.00069,0.641,0.782,0.6520,0.0767,-6.228,0.0456,93.006,0.376


In [62]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [63]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Scaled feature matrix shape:", X_scaled.shape)

Scaled feature matrix shape: (18542, 9)


In [64]:
knn = NearestNeighbors(
    n_neighbors=20,
    metric="cosine",
    algorithm="brute"
)

knn.fit(X_scaled)

print("KNN model created successfully.")

KNN model created successfully.


In [65]:
def recommend_from_features(
    acousticness,
    danceability,
    energy,
    instrumentalness,
    liveness,
    loudness,
    speechiness,
    tempo,
    valence,
    n_recommendations=5
):
    
    # Create a DataFrame containing the new song
    new_song = pd.DataFrame([{
        "acousticness": acousticness,
        "danceability": danceability,
        "energy": energy,
        "instrumentalness": instrumentalness,
        "liveness": liveness,
        "loudness": loudness,
        "speechiness": speechiness,
        "tempo": tempo,
        "valence": valence
    }])
    
    # Scale using the SAME scaler used during model preparation
    new_song_scaled = scaler.transform(new_song[feature_columns])
    
    # Find nearest songs
    distances, indices = knn.kneighbors(
        new_song_scaled,
        n_neighbors=n_recommendations
    )
    
    # Build result
    recommendations = df_filtered.iloc[indices[0]].copy()
    
    # Convert cosine distance to cosine similarity
    recommendations["similarity"] = 1 - distances[0]
    
    # Select useful columns
    result_columns = [
        "track_id",
        "track_name",
        "artist_name",
        "album_name",
        "year",
        "language",
        "popularity",
        "similarity"
    ]
    
    return recommendations[result_columns].reset_index(drop=True)

In [66]:
recommendations = recommend_from_features(
    acousticness=0.0725,
    danceability=0.676,
    energy=0.664,
    instrumentalness=5.65e-05,
    liveness=0.26,
    loudness=-5.919,
    speechiness=0.0864,
    tempo=123.455,
    valence=0.847,
    n_recommendations=10
)

display(recommendations)

,track_id,track_name,artist_name,album_name,year,language,popularity,similarity
0,3ToWC9JYSmDIa9yJs0k6PO,Te Dejo Madrid,Shakira,Laundry Service,2001,English,54,0.989135
1,2SWqtM6mPpTonFbzxyHfTZ,Celebration (feat. Akon),"Madonna, Akon",Revolver,2009,English,31,0.985334
2,6Yt72tcPl1yWaT6mL8BqF9,Il-Kimbu,Soċjetà Mużikali Madonna tal-Ġilju,Marċi Brijużi - Volume 3 - 2005,2022,English,0,0.981559
3,5Ku2CP9ansYVpAHsmmp94Z,Sweet Dreams - Avicii Swede Radio Mix,Avicii,Sweet Dreams,2011,English,7,0.980897
4,7FA6MN0Pm4cJToenMj4EK8,Jaane Kyun - Sped Up,"Vishal-Shekhar, Vishal Dadlani, Bollywood Sped Up",Jaane Kyun (Sped Up),2024,Hindi,30,0.979700
5,6gTREVvbhMEIluqtLRMWuI,Lucky Star - New Mix,Madonna,Madonna,1983,English,31,0.975630
6,2QOCuy2yCXiuDd2dHDIAnH,Elephant Gun - Remaster 2022,House of Shakira,Lint XXV,1997,English,0,0.973009
7,1cvOL5YMtugK7BzubpICYN,Sultan Mashup,"Vishal-Shekhar, Shekhar Ravjiani, Vishal Dadla...",Sultan Mashup,2016,Hindi,10,0.972280
8,6gcA4zzG6FnMEgPSDKbAMs,Been You,Justin Bieber,Purpose (Deluxe),2015,English,53,0.971567
9,3BovdzfaX4jb5KFQwoPfAw,Beat It,Michael Jackson,Thriller,1982,English,77,0.971095


array([[-0.7340576 ,  1.21677983, -0.18787721, ...,  1.28835819,
         0.11909945,  1.48979198],
       [-0.24654792,  0.65140685,  0.79977598, ..., -0.30777009,
        -0.36664419,  0.12194795],
       [ 0.27124187,  0.28898827, -0.23578576, ...,  0.07894731,
        -0.60516035,  1.10901378],
       ...,
       [ 0.77691961,  0.82536777, -1.6800443 , ...,  6.04512923,
         0.04101111,  0.19958234],
       [ 0.84353584, -0.17490751, -0.12522757, ..., -0.36732163,
         1.34237894, -0.36234277],
       [ 1.05852458,  0.41462671, -1.3966468 , ...,  6.18481802,
        -1.153974  , -0.21446774]])

In [ ]:
def recommend_from_dict(
    song_features,
    language,
    n_recommendations=10
):
    
    # Check that all required audio features exist
    missing_features = [
        feature
        for feature in feature_columns
        if feature not in song_features
    ]
    
    if missing_features:
        raise ValueError(
            f"Missing features: {missing_features}"
        )
    
    # Check that language exists
    if language not in df["language"].unique():
        raise ValueError(
            f"Language '{language}' not found in dataset."
        )
    
    # Create input DataFrame
    new_song = pd.DataFrame([{
        feature: song_features[feature]
        for feature in feature_columns
    }])
    
    # Check for missing values
    if new_song.isnull().any().any():
        raise ValueError(
            "Input contains missing values."
        )
    
    # Scale using the already fitted scaler
    new_song_scaled = scaler.transform(new_song)
    
    # Filter catalog by language BEFORE finding recommendations
    language_df = df[
        df["language"] == language
    ].copy()
    
    if len(language_df) < n_recommendations:
        raise ValueError(
            f"Only {len(language_df)} songs available "
            f"for language '{language}'."
        )
    
    # Get positions of the language-filtered songs
    language_indices = language_df.index.to_numpy()
    
    language_X_scaled = X_scaled[language_indices]
    
    # Create a temporary KNN search over only this language
    language_knn = NearestNeighbors(
        n_neighbors=n_recommendations,
        metric="cosine",
        algorithm="brute"
    )
    
    language_knn.fit(language_X_scaled)
    
    # Find nearest songs
    distances, indices = language_knn.kneighbors(
        new_song_scaled,
        n_neighbors=n_recommendations
    )
    
    # Convert local indices back to original dataframe indices
    original_indices = language_indices[indices[0]]
    
    recommendations = df.iloc[
        original_indices
    ].copy()
    
    # Convert cosine distance to similarity
    recommendations["similarity"] = (
        1 - distances[0]
    )
    
    # Output columns
    result_columns = [
        "track_id",
        "track_name",
        "artist_name",
        "album_name",
        "year",
        "language",
        "popularity",
        "similarity"
    ]
    
    return recommendations[
        result_columns
    ].reset_index(drop=True)